In [1]:
from ultralytics import YOLO
model = YOLO(r'E:\ActionRecognition\behavionRecognition-src\yolo26x.pt').to('cuda')  # 加载预训练的 YOLO 模型
result = model(r'E:\ActionRecognition\behavionRecognition-src\dataset\raw_videos\abnormal\重制数据集-merged\normal\normal_sneeze_dog\vomit_dog_9.mp4',
               save=True, 
               max_det=1,
               classes=[15, 16]
)


d:\Anaconda\envs\AR\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]



WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/212) E:\ActionRecognition\behavionRecognition-src\dataset\raw_videos\abnormal\-merged\normal\normal_sneeze_dog\vomit_dog_9.mp4: 384x640 (no detections), 50.2ms
video 1/1 (frame 2/212) E:\ActionRecognition\behavionRecognition-src\dataset\raw_videos\abnormal\-merged\normal\normal_sneeze_dog\vomit_dog_9.mp4: 384x640 1 cat, 11.5ms
video 1/1 (frame 3/212) E:\ActionRecognition\behavionRecognition-src\dataset\raw_videos\abnormal\-merged\normal

In [ ]:
import argparse
import random
import shutil
from pathlib import Path


def move_train_to_val(dataset_root: Path, ratio: float = 0.2, seed: int = 42):
    """
    目录结构示例：
    dataset_root/
      convulsion/
        train/
        val/
      limp/
        train/
        val/
      ...
    """
    random.seed(seed)

    if not dataset_root.exists():
        raise FileNotFoundError(f"数据集根目录不存在: {dataset_root}")

    class_dirs = [p for p in dataset_root.iterdir() if p.is_dir()]
    if not class_dirs:
        print(f"[WARN] 根目录下没有类别子目录: {dataset_root}")
        return

    total_moved = 0

    for cls_dir in sorted(class_dirs):
        train_dir = cls_dir / "train"
        val_dir = cls_dir / "val"

        if not train_dir.exists() or not train_dir.is_dir():
            print(f"[SKIP] {cls_dir.name}: 未找到 train 目录")
            continue

        val_dir.mkdir(parents=True, exist_ok=True)

        # 样本级别：train 下的“直接子项”（可为文件夹或文件）
        samples = [p for p in train_dir.iterdir() if p.name not in {".DS_Store"}]
        n_total = len(samples)

        if n_total == 0:
            print(f"[SKIP] {cls_dir.name}: train 为空")
            continue

        n_move = int(n_total * ratio)
        if n_move == 0 and n_total > 0 and ratio > 0:
            n_move = 1  # 可按需删除这一行，避免小样本时一个都不移动

        selected = random.sample(samples, n_move) if n_move > 0 else []

        moved_this_class = 0
        for src in selected:
            dst = val_dir / src.name

            # 防重名：若 val 已有同名，自动加后缀
            if dst.exists():
                stem = src.stem
                suffix = src.suffix
                idx = 1
                while True:
                    candidate = val_dir / f"{stem}__mv{idx}{suffix}"
                    if not candidate.exists():
                        dst = candidate
                        break
                    idx += 1

            shutil.move(str(src), str(dst))
            moved_this_class += 1

        total_moved += moved_this_class
        print(
            f"[OK] {cls_dir.name}: train={n_total} -> move={moved_this_class}, "
            f"remain_train={n_total - moved_this_class}, val_add={moved_this_class}"
        )

    print(f"\n完成，总移动样本数: {total_moved}")


if __name__ == "__main__":
    root = r'..\dataset\abnormal\5_cls\a'
    ratio = 0.2
    move_train_to_val(Path(root), ratio=ratio)

[OK] convulsion: train=684 -> move=136, remain_train=548, val_add=136

完成，总移动样本数: 136


In [3]:
import argparse
from pathlib import Path

LABEL_MAP = {
    "normal": 0,
    "convulsion": 1,
    "limp": 2,
    "sneeze": 3,
    "vomit": 4,
}


def fix_labels(dataset_root: Path, dry_run: bool = False):
    """
    目录示例:
    dataset_root/
      train/
        sneeze_cat_6.mp4_0/
          frame_0000.jpg
          label.txt
      val/
        ...
    """
    if not dataset_root.exists():
        raise FileNotFoundError(f"根目录不存在: {dataset_root}")

    total_samples = 0
    updated = 0
    skipped = 0
    missing_label = 0

    for split in ("train", "val"):
        split_dir = dataset_root / split
        if not split_dir.exists():
            print(f"[SKIP] 不存在目录: {split_dir}")
            continue

        # 只处理样本子目录
        for sample_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            total_samples += 1
            sample_name = sample_dir.name  # 例如 sneeze_cat_6.mp4_0
            cls_name = sample_name.split("_", 1)[0].lower()

            if cls_name not in LABEL_MAP:
                print(f"[WARN] 未知类别，跳过: {sample_name} (解析得到: {cls_name})")
                skipped += 1
                continue

            target_label = str(LABEL_MAP[cls_name])
            label_file = sample_dir / "label.txt"

            if not label_file.exists():
                print(f"[MISS] 缺少 label.txt: {sample_dir}")
                missing_label += 1
                if not dry_run:
                    label_file.write_text(target_label, encoding="utf-8")
                    print(f"       已新建并写入: {target_label}")
                    updated += 1
                continue

            old = label_file.read_text(encoding="utf-8").strip()
            if old != target_label:
                print(f"[FIX] {sample_name}: {old} -> {target_label}")
                if not dry_run:
                    label_file.write_text(target_label, encoding="utf-8")
                updated += 1

    print("\n===== 修正完成 =====")
    print(f"总样本目录: {total_samples}")
    print(f"已更新/新建: {updated}")
    print(f"未知类别跳过: {skipped}")
    print(f"缺失label.txt: {missing_label}")


if __name__ == "__main__":
    root = r'..\dataset\abnormal\5_cls'
    dry_run=False
    fix_labels(Path(root), dry_run=dry_run)

[FIX] convulsion_cat_1.mp4_0: -1 -> 1
[FIX] convulsion_cat_10.mp4_0: -1 -> 1
[FIX] convulsion_cat_10.mp4_1: -1 -> 1
[FIX] convulsion_cat_101.mp4_0: -1 -> 1
[FIX] convulsion_cat_101.mp4_1: -1 -> 1
[FIX] convulsion_cat_101.mp4_3: -1 -> 1
[FIX] convulsion_cat_102.mp4_0: -1 -> 1
[FIX] convulsion_cat_102.mp4_1: -1 -> 1
[FIX] convulsion_cat_102.mp4_2: -1 -> 1
[FIX] convulsion_cat_103.mp4_0: -1 -> 1
[FIX] convulsion_cat_103.mp4_1: -1 -> 1
[FIX] convulsion_cat_104.mp4_0: -1 -> 1
[FIX] convulsion_cat_105.mp4_0: -1 -> 1
[FIX] convulsion_cat_106.mp4_0: -1 -> 1
[FIX] convulsion_cat_106.mp4_1: -1 -> 1
[FIX] convulsion_cat_107.mp4_0: -1 -> 1
[FIX] convulsion_cat_107.mp4_1: -1 -> 1
[FIX] convulsion_cat_108.mp4_0: -1 -> 1
[FIX] convulsion_cat_109.mp4_0: -1 -> 1
[FIX] convulsion_cat_109.mp4_1: -1 -> 1
[FIX] convulsion_cat_11.mp4_0: -1 -> 1
[FIX] convulsion_cat_11.mp4_1: -1 -> 1
[FIX] convulsion_cat_11.mp4_3: -1 -> 1
[FIX] convulsion_cat_110.mp4_1: -1 -> 1
[FIX] convulsion_cat_110.mp4_2: -1 -> 1
[FIX] c